In [2]:
!pip install playwright

# Test Scrap

In [3]:
import requests
import pandas as pd
import time

def scrape_sikumbang_with_details(page_number=1, limit_per_page=5):
    search_url = "https://sikumbang.tapera.go.id/ajax/lokasi/search"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Referer": "https://sikumbang.tapera.go.id/",
        "X-Requested-With": "XMLHttpRequest"
    }

    params = {
        "sort": "terbaru",
        "page": page_number,
        "limit": limit_per_page
    }

    print(f"Mengambil daftar lokasi dari halaman {page_number}...")

    try:
        response = requests.get(search_url, params=params, headers=headers, timeout=10)
        response.raise_for_status()

        data_list = response.json().get('data', [])
        all_rows = []

        for lokasi in data_list:
            id_lokasi = lokasi.get("idLokasi")
            nama_perum = lokasi.get("namaPerumahan")

            print(f" -> Menarik detail unit untuk: {nama_perum} ({id_lokasi})")

            detail_url = f"https://sikumbang.tapera.go.id/lokasi-perumahan/{id_lokasi}/json"

            try:
                detail_resp = requests.get(detail_url, headers=headers, timeout=10)
                if detail_resp.status_code != 200:
                    continue

                detail_json = detail_resp.json()
                detail_info = detail_json.get("detail", {})

                kantor_list = detail_info.get("kantorPemasaran", [])
                kontak_utama = kantor_list[0] if kantor_list else {}

                bangunan_list = detail_json.get("bangunan", [])

                info_lokasi = {
                    "id_lokasi": id_lokasi,
                    "nama_perumahan": detail_info.get("namaPerumahan"),
                    "alamat": kontak_utama.get("alamat"),
                    "telpon": kontak_utama.get("noTelp"),
                    "email": kontak_utama.get("email"),
                    "website": kontak_utama.get("website"),
                    "provinsi": detail_info.get("wilayah", {}).get("provinsi"),
                    "kabupaten": detail_info.get("wilayah", {}).get("kabupaten"),
                    "kecamatan": detail_info.get("wilayah", {}).get("kecamatan"),
                    "koordinat": detail_info.get("koordinatPerumahan"),
                    "pengembang": detail_info.get("pengembang", {}).get("nama"),
                }

                for bgn in bangunan_list:
                    tipe = bgn.get("tipe", {})
                    created_at_raw = tipe.get("createdAt")

                    row = {
                        **info_lokasi,
                        "id_rumah": bgn.get("idRumah"),
                        "blok": bgn.get("blok", {}).get("blok") if bgn.get("blok") else None,
                        "nomor_rumah": bgn.get("nomor"),
                        "tipe_bangunan": bgn.get("tipeBangunan"),
                        "status_unit": bgn.get("status"),
                        "npwp_MK": bgn.get("npwpMK"),
                        "nik_pemilik": bgn.get("nikPemilik"),
                        "nik_booking": bgn.get("nikBooking"),
                        "tanggal_terjual": bgn.get("tanggalTerjual"),

                        "nama_tipe": tipe.get("nama"),
                        "harga": tipe.get("harga"),
                        "luas_bangunan": tipe.get("luasBangunan"),
                        "luas_lahan": tipe.get("luasTanah"),
                        "kamar_tidur": tipe.get("kamarTidur"),
                        "kamar_mandi": tipe.get("kamarMandi"),
                        "atap": tipe.get("spesifikasiAtap"),
                        "dinding": tipe.get("spesifikasiDinding"),
                        "lantai": tipe.get("spesifikasiLantai"),
                        "pondasi": tipe.get("spesifikasiPondasi"),

                        # Ekstraksi waktu dari tipe rumah
                        "createdAt": created_at_raw,
                        "updatedAt": tipe.get("updatedAt"),
                        "tahun": created_at_raw[:4] if created_at_raw else None
                    }
                    all_rows.append(row)

                time.sleep(0.5)

            except Exception as e:
                print(f"Gagal mengambil detail {id_lokasi}: {e}")
                continue

        df = pd.DataFrame(all_rows)

        # Konversi semua kolom yang berhubungan dengan tanggal menggunakan datetime
        if not df.empty:
            if 'tanggal_terjual' in df.columns:
                df['tanggal_terjual'] = pd.to_datetime(df['tanggal_terjual'], errors='coerce').dt.date
            if 'createdAt' in df.columns:
                df['createdAt'] = pd.to_datetime(df['createdAt'], errors='coerce').dt.date
            if 'updatedAt' in df.columns:
                df['updatedAt'] = pd.to_datetime(df['updatedAt'], errors='coerce').dt.date

        return df

    except Exception as e:
        print(f"Terjadi kesalahan utama: {e}")
        return pd.DataFrame()

# Eksekusi
df_test = scrape_sikumbang_with_details(page_number=1, limit_per_page=5)

if not df_test.empty:
    print(f"\nBerhasil mengambil {len(df_test)} unit rumah secara individual.")

    # Pratinjau untuk memastikan kolom tanggal baru sudah masuk dan diformat dengan benar
    kolom_cek_tanggal = ['nama_perumahan', 'blok', 'nomor_rumah', 'createdAt', 'updatedAt', 'tahun']

    print("\n--- Pratinjau Kolom Waktu (CreatedAt & UpdatedAt) ---")
    print(df_test[kolom_cek_tanggal].head(10))

Mengambil daftar lokasi dari halaman 1...
 -> Menarik detail unit untuk: Perumahan Rolanda CRB (MTP0520342026T005)
 -> Menarik detail unit untuk: Graha Nuansa Indramayu (IDM1510072026T001)
 -> Menarik detail unit untuk: GRESIK MODERN RESIDENCE (GSK1020182026T002)
 -> Menarik detail unit untuk: THE SCANDI 2 (LBP0620012026T003)
 -> Menarik detail unit untuk: Eito Banjaran Residency (LBP0720132026T002)

Berhasil mengambil 32 unit rumah secara individual.

--- Pratinjau Kolom Waktu (CreatedAt & UpdatedAt) ---
          nama_perumahan blok nomor_rumah   createdAt   updatedAt tahun
0  Perumahan Rolanda CRB    A           1  2026-05-11  2026-05-11  2026
1  Perumahan Rolanda CRB    A           3  2026-05-11  2026-05-11  2026
2  Perumahan Rolanda CRB    B          12  2026-05-11  2026-05-11  2026
3  Perumahan Rolanda CRB    B          13  2026-05-11  2026-05-11  2026
4  Perumahan Rolanda CRB    B          14  2026-05-11  2026-05-11  2026
5  Perumahan Rolanda CRB    B          19  2026-05-11  20

In [6]:
df_test.groupby('nama_perumahan').head(2)

,id_lokasi,nama_perumahan,alamat,telpon,email,website,provinsi,kabupaten,kecamatan,koordinat,...,luas_lahan,kamar_tidur,kamar_mandi,atap,dinding,lantai,pondasi,createdAt,updatedAt,tahun
0,MTP0520342026T005,Perumahan Rolanda CRB,Jl. Simpang Tiga Batung RT. 003 RW 002,082351607757,rolanda.group@gmail.com,,KALIMANTAN SELATAN,KAB BANJAR,Martapura,"-3.4117130555555555,114.79666944444445",...,105,1,1,Genteng Alumunium,Batako Plester,Keramik 40 x 40,Batu Gunung,2026-05-11,2026-05-11,2026
1,MTP0520342026T005,Perumahan Rolanda CRB,Jl. Simpang Tiga Batung RT. 003 RW 002,082351607757,rolanda.group@gmail.com,,KALIMANTAN SELATAN,KAB BANJAR,Martapura,"-3.4117130555555555,114.79666944444445",...,105,1,1,Genteng Alumunium,Batako Plester,Keramik 40 x 40,Batu Gunung,2026-05-11,2026-05-11,2026
10,IDM1510072026T001,Graha Nuansa Indramayu,JL. Gatot Subroto,081222048248,nuansasuksespropertindo@gmail.com,,JAWA BARAT,KAB INDRAMAYU,Indramayu,"-6.3269836,108.3371145",...,60,2,1,Rangka baja ringan + genteng beton,hebel + plester aci,keramik 40x40,batu belah,2026-05-12,2026-05-12,2026
11,IDM1510072026T001,Graha Nuansa Indramayu,JL. Gatot Subroto,081222048248,nuansasuksespropertindo@gmail.com,,JAWA BARAT,KAB INDRAMAYU,Indramayu,"-6.3269836,108.3371145",...,60,2,1,Rangka baja ringan + genteng beton,hebel + plester aci,keramik 40x40,batu belah,2026-05-12,2026-05-12,2026
17,GSK1020182026T002,GRESIK MODERN RESIDENCE,Perumahan Gresik Modern Residence 1 K.H Syafi'i,081335001970,modernjayaland@gmail.com,https://www.modernjayaland.com/,JAWA TIMUR,KAB GRESIK,Manyar,"-7.155052777777778,112.5949",...,66,2,1,RANGKA ATAP BAJA RINGAN GENTONG BETON,HEBEL PLASTER FINISHING CAT,GRANIT,BATU KEWAL DENGAN STROUS KEDALAMAM 2M DAN SLOO...,2026-05-12,2026-05-12,2026
18,GSK1020182026T002,GRESIK MODERN RESIDENCE,Perumahan Gresik Modern Residence 1 K.H Syafi'i,081335001970,modernjayaland@gmail.com,https://www.modernjayaland.com/,JAWA TIMUR,KAB GRESIK,Manyar,"-7.155052777777778,112.5949",...,66,2,1,RANGKA ATAP BAJA RINGAN GENTONG BETON,HEBEL PLASTER FINISHING CAT,GRANIT,BATU KEWAL DENGAN STROUS KEDALAMAM 2M DAN SLOO...,2026-05-12,2026-05-12,2026


# Scrap 2025 - 2026

In [ ]:
import requests
import pandas as pd
import time

def scrape_sikumbang_details_2025_2026(limit_per_page=50):
    search_url = "https://sikumbang.tapera.go.id/ajax/lokasi/search"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
        "Referer": "https://sikumbang.tapera.go.id/",
        "X-Requested-With": "XMLHttpRequest"
    }

    page_number = 1
    all_rows = []

    target_years = [2025, 2026]
    min_year = min(target_years)
    max_year = max(target_years)

    print(f"Memulai pengambilan data unit detail untuk tahun {min_year} hingga {max_year}...")

    stop_scraping = False

    while not stop_scraping:
        params = {
            "sort": "terbaru",
            "page": page_number,
            "limit": limit_per_page
        }

        print(f"\n[INFO] Mengambil daftar lokasi dari halaman {page_number}...")

        try:
            response = requests.get(search_url, params=params, headers=headers, timeout=15)
            response.raise_for_status()

            data_list = response.json().get('data', [])

            # Jika halaman kosong, proses selesai
            if not data_list:
                print("Semua data telah berhasil ditarik.")
                break

            for lokasi in data_list:
                id_lokasi = lokasi.get("idLokasi")
                nama_perum = lokasi.get("namaPerumahan")

                detail_url = f"https://sikumbang.tapera.go.id/lokasi-perumahan/{id_lokasi}/json"

                try:
                    detail_resp = requests.get(detail_url, headers=headers, timeout=10)
                    if detail_resp.status_code != 200:
                        continue

                    detail_json = detail_resp.json()
                    detail_info = detail_json.get("detail", {})

                    kantor_list = detail_info.get("kantorPemasaran", [])
                    kontak_utama = kantor_list[0] if kantor_list else {}

                    bangunan_list = detail_json.get("bangunan", [])

                    info_lokasi = {
                        "id_lokasi": id_lokasi,
                        "nama_perumahan": detail_info.get("namaPerumahan"),
                        "alamat": kontak_utama.get("alamat"),
                        "telpon": kontak_utama.get("noTelp"),
                        "email": kontak_utama.get("email"),
                        "website": kontak_utama.get("website"),
                        "provinsi": detail_info.get("wilayah", {}).get("provinsi"),
                        "kabupaten": detail_info.get("wilayah", {}).get("kabupaten"),
                        "kecamatan": detail_info.get("wilayah", {}).get("kecamatan"),
                        "koordinat": detail_info.get("koordinatPerumahan"),
                        "pengembang": detail_info.get("pengembang", {}).get("nama"),
                    }

                    for bgn in bangunan_list:
                        tipe = bgn.get("tipe", {})
                        created_at_raw = tipe.get("createdAt")

                        if not created_at_raw:
                            continue

                        # Ekstraksi Tahun
                        date_obj = pd.to_datetime(created_at_raw)
                        current_year = date_obj.year

                        # Logika Filter Tahun
                        if current_year > max_year:
                            continue # Skip jika > 2026

                        elif current_year in target_years:
                            row = {
                                **info_lokasi,
                                "id_rumah": bgn.get("idRumah"),
                                "blok": bgn.get("blok", {}).get("blok") if bgn.get("blok") else None,
                                "nomor_rumah": bgn.get("nomor"),
                                "tipe_bangunan": bgn.get("tipeBangunan"),
                                "status_unit": bgn.get("status"),
                                "npwp_MK": bgn.get("npwpMK"),
                                "nik_pemilik": bgn.get("nikPemilik"),
                                "nik_booking": bgn.get("nikBooking"),
                                "tanggal_terjual": bgn.get("tanggalTerjual"),

                                "nama_tipe": tipe.get("nama"),
                                "harga": tipe.get("harga"),
                                "luas_bangunan": tipe.get("luasBangunan"),
                                "luas_lahan": tipe.get("luasTanah"),
                                "kamar_tidur": tipe.get("kamarTidur"),
                                "kamar_mandi": tipe.get("kamarMandi"),
                                "atap": tipe.get("spesifikasiAtap"),
                                "dinding": tipe.get("spesifikasiDinding"),
                                "lantai": tipe.get("spesifikasiLantai"),
                                "pondasi": tipe.get("spesifikasiPondasi"),

                                "createdAt": created_at_raw,
                                "updatedAt": tipe.get("updatedAt"),
                                "tahun": current_year
                            }
                            all_rows.append(row)

                        elif current_year < min_year:
                            # Berhenti jika menyentuh data tahun 2024 atau sebelumnya
                            stop_scraping = True
                            break

                    if stop_scraping:
                        break # Hentikan iterasi lokasi

                    # Print progress (opsional agar terminal tidak terlalu sepi saat menarik data banyak)
                    print(f" -> {nama_perum}: OK")
                    time.sleep(0.5)

                except Exception as e:
                    print(f"Gagal mengambil detail {id_lokasi}: {e}")
                    continue

            if stop_scraping:
                print(f"\n[INFO] Data tahun {min_year-1} terdeteksi. Menghentikan pencarian lebih lanjut.")
                break # Hentikan while loop utama

            print(f"Halaman {page_number} selesai. Total unit yang relevan terkumpul sejauh ini: {len(all_rows)}")
            page_number += 1

        except Exception as e:
            print(f"Terjadi kesalahan utama pada halaman {page_number}: {e}")
            break

    df = pd.DataFrame(all_rows)

    # Konversi Format Tanggal
    if not df.empty:
        if 'tanggal_terjual' in df.columns:
            df['tanggal_terjual'] = pd.to_datetime(df['tanggal_terjual'], errors='coerce').dt.date
        if 'createdAt' in df.columns:
            df['createdAt'] = pd.to_datetime(df['createdAt'], errors='coerce').dt.date
        if 'updatedAt' in df.columns:
            df['updatedAt'] = pd.to_datetime(df['updatedAt'], errors='coerce').dt.date

    return df

# Eksekusi
# Limit dikurangi ke 50 per halaman untuk menghindari timeout, karena setiap baris lokasi memicu 1 request detail tambahan.
df_hasil = scrape_sikumbang_details_2025_2026(limit_per_page=50)

if not df_hasil.empty:
    print(f"\nBerhasil mengambil {len(df_hasil)} detail unit rumah untuk tahun 2025-2026.")

    filename = "data_tapera_detail_unit_2025_2026.csv"
    df_hasil.to_csv(filename, index=False)
    print(f"Data disimpan ke {filename}")
else:
    print("\nTidak ada data yang sesuai dengan rentang tahun tersebut.")

Memulai pengambilan data unit detail untuk tahun 2025 hingga 2026...

[INFO] Mengambil daftar lokasi dari halaman 1...
 -> KARANGSARI RESIDENCE: OK
 -> RAN SHAQUEENA RESIDENT TAHAP IV: OK
 -> GRAHA KHAYANGAN HILLS IV: OK
 -> THE HALONA TAMANYELENG: OK
 -> GREEN NARAYA RESIDENCE: OK
 -> Tirta Harmoni: OK
 -> Banteng Residence D: OK
 -> ANAIWOI VILLAGE RESIDENCE II: OK
 -> BUKIT MUTIARA ESTATE: OK
 -> WMP9 SUTARTI: OK
 -> Griya Sakinah 2: OK
 -> CITRA PERMATA 3: OK
 -> GRIYA ARYALOKA SUKOREJO: OK
 -> ARSOL LAND TAHAP 2: OK
 -> GRAHA PERINTIS PERMAI II: OK
 -> BTN AIK DAREK INDAH: OK
 -> GRIYA CITRA LEPO-LEPO: OK
 -> GRIIYA TAMAN KAKTUS CLUSTER B: OK
 -> PERMATA CITRA RESIDENCE: OK
 -> GRAND AZKIA ZAMZAM: OK
 -> ERLANGGA MEGA RESIDENCE - BUKIT SULAA: OK
 -> GRAHA STAVIA: OK
 -> Permata Indah 2: OK
 -> SUKASARI RESIDENCE: OK
 -> WMP HAPPY BARAT: OK
 -> Griya Indah: OK
 -> KALUNA UTAMA RESIDENCE: OK
 -> GRIYA ANUGERAH 1: OK
 -> BINTANG REGENCY IV: OK
 -> ISABELLA RESIDENCE: OK
 -> AZFAR IND

In [ ]:
df_hasil.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94421 entries, 0 to 94420
Data columns (total 33 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_lokasi        94421 non-null  object 
 1   nama_perumahan   94421 non-null  object 
 2   alamat           94376 non-null  object 
 3   telpon           94376 non-null  object 
 4   email            94376 non-null  object 
 5   website          94376 non-null  object 
 6   provinsi         94421 non-null  object 
 7   kabupaten        94421 non-null  object 
 8   kecamatan        94421 non-null  object 
 9   koordinat        94421 non-null  object 
 10  pengembang       94421 non-null  object 
 11  id_rumah         94421 non-null  object 
 12  blok             94197 non-null  object 
 13  nomor_rumah      94421 non-null  object 
 14  tipe_bangunan    94421 non-null  object 
 15  status_unit      94421 non-null  object 
 16  npwp_MK          94421 non-null  object 
 17  nik_pemilik 

In [ ]:
# Simpan df_test ke Google Drive
from google.colab import drive
drive.mount('/content/drive')


output_path = '/content/drive/MyDrive/df_hasil_scrap_sikumbang_2025_2026.csv'
df_hasil.to_csv(output_path, index=False)

print(f"DataFrame 'df_hasil' berhasil disimpan ke {output_path}")

Mounted at /content/drive
DataFrame 'df_hasil' berhasil disimpan ke /content/drive/MyDrive/df_hasil_scrap_sikumbang_2025_2026.csv


# Scrap 1 Bulan At 2026

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime

def scrape_sikumbang_details_by_month(target_year=2026, target_month=4, limit_per_page=50):
    search_url = "https://sikumbang.tapera.go.id/ajax/lokasi/search"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Referer": "https://sikumbang.tapera.go.id/",
        "X-Requested-With": "XMLHttpRequest"
    }

    page_number = 1
    all_rows = []

    print(f"Memulai pengambilan data detail unit untuk periode {target_year}-{target_month:02d}...")

    stop_scraping = False

    while not stop_scraping:
        params = {
            "sort": "terbaru",
            "page": page_number,
            "limit": limit_per_page
        }

        print(f"\n[INFO] Mengambil daftar lokasi dari halaman {page_number}...")

        try:
            response = requests.get(search_url, params=params, headers=headers, timeout=15)
            response.raise_for_status()

            data_list = response.json().get('data', [])

            # Jika halaman kosong, proses selesai
            if not data_list:
                print("Semua data telah berhasil ditarik.")
                break

            for lokasi in data_list:
                id_lokasi = lokasi.get("idLokasi")
                nama_perum = lokasi.get("namaPerumahan")

                detail_url = f"https://sikumbang.tapera.go.id/lokasi-perumahan/{id_lokasi}/json"

                try:
                    detail_resp = requests.get(detail_url, headers=headers, timeout=10)
                    if detail_resp.status_code != 200:
                        continue

                    detail_json = detail_resp.json()
                    detail_info = detail_json.get("detail", {})

                    kantor_list = detail_info.get("kantorPemasaran", [])
                    kontak_utama = kantor_list[0] if kantor_list else {}

                    bangunan_list = detail_json.get("bangunan", [])

                    info_lokasi = {
                        "id_lokasi": id_lokasi,
                        "nama_perumahan": detail_info.get("namaPerumahan"),
                        "alamat": kontak_utama.get("alamat"),
                        "telpon": kontak_utama.get("noTelp"),
                        "email": kontak_utama.get("email"),
                        "website": kontak_utama.get("website"),
                        "provinsi": detail_info.get("wilayah", {}).get("provinsi"),
                        "kabupaten": detail_info.get("wilayah", {}).get("kabupaten"),
                        "kecamatan": detail_info.get("wilayah", {}).get("kecamatan"),
                        "koordinat": detail_info.get("koordinatPerumahan"),
                        "pengembang": detail_info.get("pengembang", {}).get("nama"),
                    }

                    for bgn in bangunan_list:
                        tipe = bgn.get("tipe", {})
                        created_at_raw = tipe.get("createdAt")

                        if not created_at_raw:
                            continue

                        # Ekstraksi Tahun dan Bulan
                        date_obj = pd.to_datetime(created_at_raw)
                        current_year = date_obj.year
                        current_month = date_obj.month

                        # 1. SKIP LOGIC: Jika data lebih baru dari target (misal target April, data ini Mei)
                        if current_year > target_year or (current_year == target_year and current_month > target_month):
                            continue

                        # 2. SAVE LOGIC: Jika data tepat pada tahun dan bulan target
                        elif current_year == target_year and current_month == target_month:
                            row = {
                                **info_lokasi,
                                "id_rumah": bgn.get("idRumah"),
                                "blok": bgn.get("blok", {}).get("blok") if bgn.get("blok") else None,
                                "nomor_rumah": bgn.get("nomor"),
                                "tipe_bangunan": bgn.get("tipeBangunan"),
                                "status_unit": bgn.get("status"),
                                "npwp_MK": bgn.get("npwpMK"),
                                "nik_pemilik": bgn.get("nikPemilik"),
                                "nik_booking": bgn.get("nikBooking"),
                                "tanggal_terjual": bgn.get("tanggalTerjual"),

                                "nama_tipe": tipe.get("nama"),
                                "harga": tipe.get("harga"),
                                "luas_bangunan": tipe.get("luasBangunan"),
                                "luas_lahan": tipe.get("luasTanah"),
                                "kamar_tidur": tipe.get("kamarTidur"),
                                "kamar_mandi": tipe.get("kamarMandi"),
                                "atap": tipe.get("spesifikasiAtap"),
                                "dinding": tipe.get("spesifikasiDinding"),
                                "lantai": tipe.get("spesifikasiLantai"),
                                "pondasi": tipe.get("spesifikasiPondasi"),

                                "createdAt": created_at_raw,
                                "updatedAt": tipe.get("updatedAt"),
                                "tahun": current_year,
                                "bulan": current_month
                            }
                            all_rows.append(row)

                        # 3. BREAK LOGIC: Jika data lebih tua dari target (misal target April, data ini Maret)
                        elif current_year < target_year or (current_year == target_year and current_month < target_month):
                            stop_scraping = True
                            break

                    if stop_scraping:
                        break # Hentikan iterasi bangunan/lokasi di halaman ini

                    # Progress indikator
                    print(f" -> {nama_perum}: OK")
                    time.sleep(0.5) # Jeda untuk menghindari rate limit API

                except Exception as e:
                    print(f"Gagal mengambil detail {id_lokasi}: {e}")
                    continue

            if stop_scraping:
                print(f"\n[INFO] Data dari bulan sebelumnya telah terdeteksi. Menghentikan pencarian lebih lanjut.")
                break # Hentikan while loop utama

            print(f"Halaman {page_number} selesai. Total unit target yang terkumpul sejauh ini: {len(all_rows)}")
            page_number += 1

        except Exception as e:
            print(f"Terjadi kesalahan utama pada halaman {page_number}: {e}")
            break

    df = pd.DataFrame(all_rows)

    # Konversi Format Tanggal agar bersih di CSV (YYYY-MM-DD)
    if not df.empty:
        if 'tanggal_terjual' in df.columns:
            df['tanggal_terjual'] = pd.to_datetime(df['tanggal_terjual'], errors='coerce').dt.date
        if 'createdAt' in df.columns:
            df['createdAt'] = pd.to_datetime(df['createdAt'], errors='coerce').dt.date
        if 'updatedAt' in df.columns:
            df['updatedAt'] = pd.to_datetime(df['updatedAt'], errors='coerce').dt.date

    return df

# Eksekusi untuk bulan April 2026
# limit_per_page di set 50 untuk mencegah putusnya koneksi karena beban request
df_april_2026 = scrape_sikumbang_details_by_month(target_year=2026, target_month=4, limit_per_page=50)

if not df_april_2026.empty:
    print(f"\nBerhasil mengambil {len(df_april_2026)} detail unit rumah untuk periode 2026-04.")

    filename = "data_tapera_detail_unit_April_2026.csv"
    df_april_2026.to_csv(filename, index=False)
    print(f"Data disimpan ke {filename}")
else:
    print("\nTidak ada data yang sesuai dengan rentang waktu tersebut.")

Memulai pengambilan data untuk periode 2026-04...
Halaman 1 diproses. Data target ditemukan: 72
Halaman 2 diproses. Data target ditemukan: 220
Halaman 3 diproses. Data target ditemukan: 330
Halaman 4 diproses. Data target ditemukan: 330

Selesai! 330 data bulan April 2026 disimpan di data_tapera_april_2026.csv


In [ ]:
# Simpan df_test ke Google Drive
output_path = '/content/drive/MyDrive/df_hasil_scrap_sikumbang_april_2026.csv'
df_april_2026.to_csv(output_path, index=False)

print(f"DataFrame 'df_april_2026' berhasil disimpan ke {output_path}")

DataFrame 'df_hasil' berhasil disimpan ke /content/drive/MyDrive/df_hasil_scrap_sikumbang_april_2026.csv


# Scrap All Data

In [ ]:
import requests
import pandas as pd
import time

def scrape_sikumbang_all_data(limit_per_page=100):
    search_url = "https://sikumbang.tapera.go.id/ajax/lokasi/search"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Referer": "https://sikumbang.tapera.go.id/",
        "X-Requested-With": "XMLHttpRequest"
    }

    page_number = 1
    all_rows = []

    print("Memulai pengambilan SELURUH data detail unit perumahan tanpa batas waktu...")
    print("Peringatan: Proses ini akan memakan waktu sangat lama karena menarik data level bangunan.")

    while True:
        params = {
            "sort": "terbaru",
            "page": page_number,
            "limit": limit_per_page
        }

        print(f"\n[INFO] Mengambil daftar lokasi dari halaman {page_number}...")

        try:
            # TAHAP 1: Ambil daftar lokasi
            response = requests.get(search_url, params=params, headers=headers, timeout=15)
            response.raise_for_status()

            data_list = response.json().get('data', [])

            # Jika halaman sudah tidak mengirimkan data, hentikan keseluruhan loop
            if not data_list:
                print("Semua halaman telah berhasil ditarik sampai tuntas.")
                break

            for lokasi in data_list:
                id_lokasi = lokasi.get("idLokasi")
                nama_perum = lokasi.get("namaPerumahan")

                # TAHAP 2: Hit detail tiap lokasi
                detail_url = f"https://sikumbang.tapera.go.id/lokasi-perumahan/{id_lokasi}/json"

                try:
                    detail_resp = requests.get(detail_url, headers=headers, timeout=10)
                    if detail_resp.status_code != 200:
                        continue

                    detail_json = detail_resp.json()
                    detail_info = detail_json.get("detail", {})

                    kantor_list = detail_info.get("kantorPemasaran", [])
                    kontak_utama = kantor_list[0] if kantor_list else {}

                    bangunan_list = detail_json.get("bangunan", [])

                    # Menghitung agregasi unit
                    jml_subsidi = 0
                    jml_subsidi_terjual = 0
                    jml_komersil = 0
                    jml_komersil_terjual = 0

                    for bgn in bangunan_list:
                        tipe_bgn = str(bgn.get("tipeBangunan")).lower()
                        status_unit = str(bgn.get("status")).lower()

                        if tipe_bgn == "subsidi":
                            jml_subsidi += 1
                            if status_unit == "terjual":
                                jml_subsidi_terjual += 1
                        elif tipe_bgn == "komersil":
                            jml_komersil += 1
                            if status_unit == "komersil-terjual":
                                jml_komersil_terjual += 1

                    info_lokasi = {
                        "id_lokasi": id_lokasi,
                        "nama_perumahan": detail_info.get("namaPerumahan"),
                        "alamat": kontak_utama.get("alamat"),
                        "telpon": kontak_utama.get("noTelp"),
                        "email": kontak_utama.get("email"),
                        "website": kontak_utama.get("website"),
                        "provinsi": detail_info.get("wilayah", {}).get("provinsi"),
                        "kabupaten": detail_info.get("wilayah", {}).get("kabupaten"),
                        "kecamatan": detail_info.get("wilayah", {}).get("kecamatan"),
                        "koordinat": detail_info.get("koordinatPerumahan"),
                        "pengembang": detail_info.get("pengembang", {}).get("nama"),
                        "jumlah_unit_subsidi": jml_subsidi,
                        "jumlah_unit_subsidi_terjual": jml_subsidi_terjual,
                        "jumlah_unit_komersil": jml_komersil,
                        "jumlah_unit_komersil_terjual": jml_komersil_terjual,
                    }

                    # TAHAP 3: Ekstrak data level bangunan
                    for bgn in bangunan_list:
                        tipe = bgn.get("tipe", {})
                        created_at_raw = tipe.get("createdAt")

                        row = {
                            **info_lokasi,
                            "id_rumah": bgn.get("idRumah"),
                            "blok": bgn.get("blok", {}).get("blok") if bgn.get("blok") else None,
                            "nomor_rumah": bgn.get("nomor"),
                            "tipe_bangunan": bgn.get("tipeBangunan"),
                            "status_unit": bgn.get("status"),
                            "npwp_MK": bgn.get("npwpMK"),
                            "nik_pemilik": bgn.get("nikPemilik"),
                            "nik_booking": bgn.get("nikBooking"),
                            "tanggal_terjual": bgn.get("tanggalTerjual"),

                            "nama_tipe": tipe.get("nama"),
                            "harga": tipe.get("harga"),
                            "luas_bangunan": tipe.get("luasBangunan"),
                            "luas_lahan": tipe.get("luasTanah"),
                            "kamar_tidur": tipe.get("kamarTidur"),
                            "kamar_mandi": tipe.get("kamarMandi"),
                            "atap": tipe.get("spesifikasiAtap"),
                            "dinding": tipe.get("spesifikasiDinding"),
                            "lantai": tipe.get("spesifikasiLantai"),
                            "pondasi": tipe.get("spesifikasiPondasi"),

                            "createdAt": created_at_raw,
                            "updatedAt": tipe.get("updatedAt"),
                            "tahun": created_at_raw[:4] if created_at_raw else None
                        }
                        all_rows.append(row)

                    # Jeda agar tidak terkena rate limit
                    time.sleep(0.5)

                except Exception as e:
                    print(f"Gagal mengambil detail {id_lokasi}: {e}")
                    continue

            print(f"Halaman {page_number} selesai. Total unit terkumpul sejauh ini: {len(all_rows)}")
            page_number += 1

        except Exception as e:
            print(f"Terjadi kesalahan utama pada halaman {page_number}: {e}")
            break

    df = pd.DataFrame(all_rows)

    # Konversi Format Tanggal
    if not df.empty:
        if 'tanggal_terjual' in df.columns:
            df['tanggal_terjual'] = pd.to_datetime(df['tanggal_terjual'], errors='coerce').dt.date
        if 'createdAt' in df.columns:
            df['createdAt'] = pd.to_datetime(df['createdAt'], errors='coerce').dt.date
        if 'updatedAt' in df.columns:
            df['updatedAt'] = pd.to_datetime(df['updatedAt'], errors='coerce').dt.date

    return df

# Eksekusi dan Simpan ke Variabel yang Diminta
# Limit di set 100 per halaman. Ini memicu 100 request detail di setiap putaran halaman.
df_script_all_data = scrape_sikumbang_all_data(limit_per_page=100)

# Simpan ke CSV
if not df_script_all_data.empty:
    print(f"\nBerhasil mengambil {len(df_script_all_data)} detail unit rumah secara keseluruhan.")
    filename = "data_tapera_all_detail_unit.csv"
    df_script_all_data.to_csv(filename, index=False)
    print(f"Data disimpan ke {filename}")
else:
    print("\nProses selesai tanpa ada data yang tersimpan.")

Memulai pengambilan seluruh data...
Halaman 1 selesai. Total data terkumpul: 96
Halaman 2 selesai. Total data terkumpul: 222
Halaman 3 selesai. Total data terkumpul: 372
Halaman 4 selesai. Total data terkumpul: 570
Halaman 5 selesai. Total data terkumpul: 819
Halaman 6 selesai. Total data terkumpul: 1136
Halaman 7 selesai. Total data terkumpul: 1405
Halaman 8 selesai. Total data terkumpul: 1577
Halaman 9 selesai. Total data terkumpul: 1832
Halaman 10 selesai. Total data terkumpul: 2060
Halaman 11 selesai. Total data terkumpul: 2260
Halaman 12 selesai. Total data terkumpul: 2471
Halaman 13 selesai. Total data terkumpul: 2729
Halaman 14 selesai. Total data terkumpul: 2914
Halaman 15 selesai. Total data terkumpul: 3064
Halaman 16 selesai. Total data terkumpul: 3442
Halaman 17 selesai. Total data terkumpul: 3613
Halaman 18 selesai. Total data terkumpul: 3898
Halaman 19 selesai. Total data terkumpul: 4108
Halaman 20 selesai. Total data terkumpul: 4282
Halaman 21 selesai. Total data terkumpu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Simpan df_test ke Google Drive
output_path = '/content/drive/MyDrive/df_hasil_scrap_sikumbang_all_data.csv'
df_script_all_data.to_csv(output_path, index=False)

print(f"DataFrame 'df_script_all_data' berhasil disimpan ke {output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DataFrame 'df_hasil' berhasil disimpan ke /content/drive/MyDrive/df_hasil_scrap_sikumbang.csv
